# L1: Create Agents to Research and Write an Article

In this lesson, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [2]:
from crewai import Agent, Task, Crew

- As a LLM for your agents, you'll be using OpenAI's `gpt-3.5-turbo`.

**Optional Note:** crewAI also allow other popular models to be used as a LLM for your Agents. You can see some of the examples at the [bottom of the notebook](#1).

In [ ]:
import os
from utils import get_openai_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-4o-mini'

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_ :
```Python
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:
```Python
varname = """line 1 of text
             line 2 of text
          """
```
is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [7]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
	verbose=True
)

### Agent: Writer

In [8]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True
)

### Agent: Editor

In [9]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [10]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [11]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

### Task: Edit

In [12]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution. 

In [14]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=True
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [ ]:
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│  Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.            │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 1a18d3cd-75e8-4304-a012-f4609683141f                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Content Plan – Artificial Intelligence Blog Article                                                            │
│                                                                                                                 │
│  1. Introduction                                                                                                │
│  - Brief overview of Artificial Intelligence (AI) and its impact on various industries.                         │
│  - Purpose of the article: to educate readers on the latest trends, key players, and news in the AI industry.   │
│                                                                                                                 │
│  2. Latest Trends in Artificial Intelligence                                                                    │
│  - Emphasis on the rise of AI-powered automation in businesses.                                                 │
│  - Exploration of the use of AI in healthcare for diagnostics and personalized treatment.                       │
│  - Discussion on the increasing integration of AI in smart home devices and autonomous vehicles.                │
│                                                                                                                 │
│  3. Key Players in the AI Industry                                                                              │
│  - Highlight of major companies leading in AI research and development, such as Google, Amazon, and Microsoft.  │
│  - Profiles of influential figures in the AI field, like Andrew Ng and Fei-Fei Li.                              │
│                                                                                                                 │
│  4. Noteworthy News in Artificial Intelligence                                                                  │
│  - Recent advancements in natural language processing and conversational AI technologies.                       │
│  - Breakthroughs in reinforcement learning and self-learning algorithms.                                        │
│  - Updates on AI ethics and regulations worldwide.                                                              │
│                                                                                                                 │
│  5. Target Audience                                                                                             │
│  - Audience: Professionals in tech industries, entrepreneurs, and general technology enthusiasts.               │
│  - Interests: Innovations in AI, market trends, career opportunities in AI-related fields.                      │
│  - Pain Points: Understanding complex AI concepts, keeping up with rapid advancements, selecting reliable AI    │
│  solutions for businesses.                                                                                      │
│                                                                                                                 │
│  6. Call to Action                                                                                              │
│  - Encourage readers to stay informed by following reputable AI news sources.                                   │
│  - Invite them to explore AI-related courses or certifications to enhance their skills.                         │
│  - Suggest subscribing to newsletters from AI industry 

Output()

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 5cad163e-2bf0-4df2-aad7-098c5db2ce66                                                                     │
│  Agent: Content Planner                                                                                         │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

Output()

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 1a18d3cd-75e8-4304-a012-f4609683141f                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: # The Power of Artificial Intelligence: Trends, Players, and News                                │
│                                                                                                                 │
│  ## Introduction                                                                                                │
│  Artificial Intelligence (AI) has emerged as a transformative technology shaping various industries with its    │
│  innovative applications. From enhancing efficiency to revolutionizing healthcare and transportation, AI is     │
│  paving the way for new possibilities. In this article, we delve into the latest trends, key players, and       │
│  noteworthy news in the ever-evolving realm of Artificial Intelligence.                                         │
│                                                                                                                 │
│  ## Latest Trends in Artificial Intelligence                                                                    │
│  One of the most significant trends in AI is the increasing use of AI-powered automation in businesses.         │
│  Companies are leveraging AI to streamline processes, enhance decision-making, and boost productivity. In       │
│  healthcare, AI plays a crucial role in diagnostics and personalized treatment, offering more accurate and      │
│  efficient healthcare solutions. Moreover, the integration of AI in smart home devices and autonomous vehicles  │
│  is reshaping the way we interact with technology, making our lives more convenient and secure.                 │
│                                                                                                                 │
│  ## Key Players in the AI Industry                                                                              │
│  Leading the frontier of AI research and development are major companies like Google, Amazon, and Microsoft.    │
│  These tech giants are investing heavily in AI to drive innovation and bring cutting-edge solutions to market.  │
│  Additionally, influential figures in the AI field such as Andrew Ng and Fei-Fei Li are spearheading            │
│  breakthroughs and inspiring the next generation of AI enthusiasts with their pioneering work.                  │
│                                                                                                                 │
│  ## Noteworthy News in Artificial Intelligence                                                                  │
│  Recent advancements in natural language processing and conversational AI technologies are revolutionizing how  │
│  we interact with machines, enabling more seamless communication and personalized experiences. Breakthroughs    │
│  in reinforcement learning and self-learning algorithms are pushing the boundaries of AI capabilities,          │
│  allowing for continuous improvement and adaptation. Furthermore, ongoing discussions on AI ethics and          │
│  regulations globally are shaping the responsible deployment of AI technologies to ensure ethical and fair      │
│  practices.                                                                                                     │
│                                                       

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 57c9dd99-1615-4001-9c07-7a637b1a0090                                                                     │
│  Agent: Editor                                                                                                  │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



╭────────────────────────────── Execution Traces ──────────────────────────────╮
│                                                                              │
│  🔍 Detailed execution traces are available!                                 │
│                                                                              │
│  View insights including:                                                    │
│    • Agent decision-making process                                           │
│    • Task execution flow and timing                                          │
│    • Tool usage details                                                      │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯
Would you like to view your execution traces? [y/N] (20s timeout): 

╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                      

- Display the results of your execution as markdown in the notebook.

In [17]:
from IPython.display import Markdown
Markdown(result.raw)

# The Power of Artificial Intelligence: Trends, Players, and News

## Introduction
Artificial Intelligence (AI) has emerged as a transformative technology shaping various industries with its innovative applications. From enhancing efficiency to revolutionizing healthcare and transportation, AI is paving the way for new possibilities. In this article, we delve into the latest trends, key players, and noteworthy news in the ever-evolving realm of Artificial Intelligence.

## Latest Trends in Artificial Intelligence
One of the most significant trends in AI is the increasing use of AI-powered automation in businesses. Companies are leveraging AI to streamline processes, enhance decision-making, and boost productivity. In healthcare, AI plays a crucial role in diagnostics and personalized treatment, offering more accurate and efficient healthcare solutions. Moreover, the integration of AI in smart home devices and autonomous vehicles is reshaping the way we interact with technology, making our lives more convenient and secure.

## Key Players in the AI Industry
Leading the frontier of AI research and development are major companies like Google, Amazon, and Microsoft. These tech giants are investing heavily in AI to drive innovation and bring cutting-edge solutions to market. Additionally, influential figures in the AI field such as Andrew Ng and Fei-Fei Li are spearheading breakthroughs and inspiring the next generation of AI enthusiasts with their pioneering work.

## Noteworthy News in Artificial Intelligence
Recent advancements in natural language processing and conversational AI technologies are revolutionizing how we interact with machines, enabling more seamless communication and personalized experiences. Breakthroughs in reinforcement learning and self-learning algorithms are pushing the boundaries of AI capabilities, allowing for continuous improvement and adaptation. Furthermore, ongoing discussions on AI ethics and regulations globally are shaping the responsible deployment of AI technologies to ensure ethical and fair practices.

## Target Audience
This article caters to professionals in tech industries, entrepreneurs, and technology enthusiasts keen on staying updated with AI advancements. Whether it's exploring innovations in AI, understanding market trends, or seeking career opportunities in AI-related fields, this content aims to provide valuable insights to address the audience's interests and pain points.

## Call to Action
To remain informed on the latest AI developments, readers are encouraged to follow reputable AI news sources for regular updates. Additionally, exploring AI-related courses or certifications can help enhance skills and stay competitive in the evolving tech landscape. Subscribing to newsletters from AI industry leaders is another way to receive curated insights and stay ahead in the dynamic AI industry.

In conclusion, Artificial Intelligence continues to redefine industries and push the boundaries of innovation. By keeping abreast of the latest trends, key players, and news in the AI landscape, individuals can position themselves for success in the AI-driven future. Stay informed, stay curious, and embrace the limitless possibilities of Artificial Intelligence.

SEO Keywords: Artificial Intelligence, AI trends, Key players in AI, AI news, Machine learning, Deep learning, Automation, AI applications, Neural networks, Robotics, Data science, Predictive analytics.

Resources: Input from industry experts, Statista, Gartner, Forbes, MIT Technology Review, IEEE.

## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [ ]:
topic = "YOUR TOPIC HERE"
result = crew.kickoff(inputs={"topic": topic})

In [ ]:
Markdown(result.raw)

<a name='1'></a>
 ## Other Popular Models as LLM for your Agents

#### Hugging Face (HuggingFaceHub endpoint)

```Python
from langchain_community.llms import HuggingFaceHub

llm = HuggingFaceHub(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    huggingfacehub_api_token="<HF_TOKEN_HERE>",
    task="text-generation",
)

### you will pass "llm" to your agent function
```

#### Mistral API

```Python
OPENAI_API_KEY=your-mistral-api-key
OPENAI_API_BASE=https://api.mistral.ai/v1
OPENAI_MODEL_NAME="mistral-small"
```

#### Cohere

```Python
from langchain_community.chat_models import ChatCohere
# Initialize language model
os.environ["COHERE_API_KEY"] = "your-cohere-api-key"
llm = ChatCohere()

### you will pass "llm" to your agent function
```

### For using Llama locally with Ollama and more, checkout the crewAI documentation on [Connecting to any LLM](https://docs.crewai.com/how-to/LLM-Connections/).